# 🏥 Fall Detection - Video Classification
**Models Compared:** Simple CNN | ResNet18 | MobileNetV2 | CNN + LSTM  
**Dataset:** Binary video classification — `fall` vs `non-fall`  
**Split:** 70% Train | 15% Validation | 15% Test

## 📦 1. Imports & Global Configuration

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import random
import time
import warnings
warnings.filterwarnings('ignore')

# ── Numerical & Visualisation ──────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models

# ── Video / Image I/O ─────────────────────────────────────────────────────────
import cv2
from PIL import Image

# ── Metrics ───────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# ── Global Hyper-parameters ───────────────────────────────────────────────────
DATA_ROOT    = 'dataset'          # Root folder: dataset/fall/, dataset/non-fall/
IMG_SIZE     = 224                # Spatial size fed to every model
NUM_FRAMES   = 16                 # Frames sampled per video (for CNN+LSTM)
BATCH_SIZE   = 32
LR           = 1e-4
NUM_EPOCHS   = 25                 # Within the 20-30 range requested
NUM_CLASSES  = 2                  # fall=1, non-fall=0
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15               # test = 1 - train - val = 0.15
RESULTS      = {}                 # Will hold per-model metrics for final table

print('Configuration loaded ✓')

## 📂 2. Dataset — Loading, Preprocessing & Splitting

**Expected folder structure**
```
dataset/
  fall/
    video_001.mp4
    video_002.avi
    ...
  non-fall/
    video_101.mp4
    ...
```
The dataset supports `.mp4`, `.avi`, `.mov`, `.mkv` files.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VideoFrameDataset  (frame-level — used by CNN, ResNet18, MobileNetV2)
# ─────────────────────────────────────────────────────────────────────────────
class VideoFrameDataset(Dataset):
    """
    Samples a single representative frame from each video.
    Suitable for frame-based CNN / transfer-learning models.
    """

    VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.wmv'}

    def __init__(self, root_dir: str, transform=None, frame_idx: int = None):
        """
        Args:
            root_dir  : Path to dataset root containing class sub-folders.
            transform : torchvision transforms applied to each frame.
            frame_idx : Fixed frame index to extract (None → random middle frame).
        """
        self.transform = transform
        self.frame_idx = frame_idx
        self.samples   = []          # List of (video_path, label)
        self.classes   = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            label   = self.class_to_idx[cls]
            for fname in os.listdir(cls_dir):
                if os.path.splitext(fname)[1].lower() in self.VIDEO_EXTS:
                    self.samples.append((os.path.join(cls_dir, fname), label))

        print(f'VideoFrameDataset: {len(self.samples)} videos | classes → {self.class_to_idx}')

    def __len__(self):
        return len(self.samples)

    def _load_frame(self, path: str) -> Image.Image:
        """Open a video and return one PIL frame."""
        cap   = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        idx   = self.frame_idx if self.frame_idx is not None else max(0, total // 2)
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        cap.release()
        if not ret or frame is None:
            # Fall-back: blank frame
            frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = self._load_frame(path)
        if self.transform:
            image = self.transform(image)
        return image, label


# ─────────────────────────────────────────────────────────────────────────────
# VideoSequenceDataset  (sequence-level — used by CNN+LSTM)
# ─────────────────────────────────────────────────────────────────────────────
class VideoSequenceDataset(Dataset):
    """
    Samples NUM_FRAMES evenly-spaced frames from each video and returns
    a tensor of shape (T, C, H, W), suitable for sequential models.
    """

    VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.wmv'}

    def __init__(self, root_dir: str, num_frames: int = NUM_FRAMES, transform=None):
        self.num_frames = num_frames
        self.transform  = transform
        self.samples    = []
        self.classes    = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            label   = self.class_to_idx[cls]
            for fname in os.listdir(cls_dir):
                if os.path.splitext(fname)[1].lower() in self.VIDEO_EXTS:
                    self.samples.append((os.path.join(cls_dir, fname), label))

        print(f'VideoSequenceDataset: {len(self.samples)} videos | classes → {self.class_to_idx}')

    def __len__(self):
        return len(self.samples)

    def _load_frames(self, path: str) -> torch.Tensor:
        """Return a tensor (T, C, H, W) of evenly-sampled frames."""
        cap    = cv2.VideoCapture(path)
        total  = max(1, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
        idxs   = np.linspace(0, total - 1, self.num_frames, dtype=int)
        frames = []
        for fi in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
            ret, frame = cap.read()
            if not ret or frame is None:
                frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if self.transform:
                pil = self.transform(pil)
            frames.append(pil)
        cap.release()
        return torch.stack(frames)   # (T, C, H, W)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        frames = self._load_frames(path)
        return frames, label


# ─────────────────────────────────────────────────────────────────────────────
# Transforms
# ─────────────────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet stats
                         std =[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

print('Transforms defined ✓')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Build datasets & dataloaders (frame-level — CNN / ResNet / MobileNet)
# ─────────────────────────────────────────────────────────────────────────────

# Full frame dataset (use val_transform; we'll manually set transform after split)
full_frame_ds = VideoFrameDataset(DATA_ROOT, transform=None)

n_total = len(full_frame_ds)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)
n_test  = n_total - n_train - n_val

print(f'Total={n_total} | Train={n_train} | Val={n_val} | Test={n_test}')

# Deterministic split (generator ensures reproducibility)
generator = torch.Generator().manual_seed(SEED)
train_ds, val_ds, test_ds = random_split(
    full_frame_ds, [n_train, n_val, n_test], generator=generator
)

# Attach appropriate transforms via Subset wrappers
class TransformSubset(Dataset):
    """Wrapper that applies a transform to a torch Subset."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        # Access the underlying VideoFrameDataset directly
        orig_idx         = self.subset.indices[idx]
        path, label      = self.subset.dataset.samples[orig_idx]
        image            = self.subset.dataset._load_frame(path)
        if self.transform:
            image = self.transform(image)
        return image, label

train_frame_ds = TransformSubset(train_ds, train_transform)
val_frame_ds   = TransformSubset(val_ds,   val_transform)
test_frame_ds  = TransformSubset(test_ds,  val_transform)

train_loader = DataLoader(train_frame_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_frame_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_frame_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('Frame-level DataLoaders ready ✓')

# ── Sequence dataloaders (CNN+LSTM) ───────────────────────────────────────────
full_seq_ds = VideoSequenceDataset(DATA_ROOT, num_frames=NUM_FRAMES, transform=None)

seq_train_ds, seq_val_ds, seq_test_ds = random_split(
    full_seq_ds, [n_train, n_val, n_test], generator=generator
)

class SeqTransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        orig_idx    = self.subset.indices[idx]
        path, label = self.subset.dataset.samples[orig_idx]
        frames      = self.subset.dataset._load_frames.__func__(
            self.subset.dataset, path
        ) if self.transform is None else self.subset.dataset._load_frames(path)
        # Re-load with the correct transform injected
        self.subset.dataset.transform = self.transform
        frames = self.subset.dataset._load_frames(path)
        return frames, label

seq_train_ds_t = SeqTransformSubset(seq_train_ds, train_transform)
seq_val_ds_t   = SeqTransformSubset(seq_val_ds,   val_transform)
seq_test_ds_t  = SeqTransformSubset(seq_test_ds,  val_transform)

seq_train_loader = DataLoader(seq_train_ds_t, batch_size=8,  shuffle=True,  num_workers=2, pin_memory=True)
seq_val_loader   = DataLoader(seq_val_ds_t,   batch_size=8,  shuffle=False, num_workers=2, pin_memory=True)
seq_test_loader  = DataLoader(seq_test_ds_t,  batch_size=8,  shuffle=False, num_workers=2, pin_memory=True)

print('Sequence-level DataLoaders ready ✓')

## 🔧 3. Shared Utilities (Training Loop & Evaluation)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Training & Evaluation Helpers
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer):
    """Run one full pass over the training set. Returns (avg_loss, accuracy)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        preds   = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader):
    """Run model over a dataloader. Returns (avg_loss, accuracy, all_preds, all_labels)."""
    model.eval()
    criterion  = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss    = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            preds   = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


def full_training_loop(model, train_loader, val_loader, model_name,
                        epochs=NUM_EPOCHS, lr=LR, save_path=None):
    """
    Complete training loop with:
      - Adam optimiser, CrossEntropyLoss
      - ReduceLROnPlateau scheduler
      - Early stopping (patience=7)
      - Best-model checkpointing
    Returns the best model state and training history dict.
    """
    criterion   = nn.CrossEntropyLoss()
    optimizer   = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler   = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

    best_val_loss   = float('inf')
    best_state      = None
    patience_count  = 0
    patience_limit  = 7
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    print(f'\n{"═"*60}')
    print(f'  Training: {model_name}')
    print(f'  Epochs={epochs} | LR={lr} | Batch={BATCH_SIZE}')
    print(f'{"═"*60}')

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader)
        scheduler.step(vl_loss)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        elapsed = time.time() - t0
        print(f'  Epoch [{epoch:02d}/{epochs}]  '
              f'Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}  │  '
              f'Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}  '
              f'({elapsed:.1f}s)')

        # Checkpoint best model
        if vl_loss < best_val_loss:
            best_val_loss  = vl_loss
            best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
            if save_path:
                torch.save(best_state, save_path)
        else:
            patience_count += 1
            if patience_count >= patience_limit:
                print(f'  ⚠ Early stopping at epoch {epoch}')
                break

    # Restore best weights
    model.load_state_dict(best_state)
    return model, history


def compute_metrics(all_preds, all_labels):
    """Compute and return a dict with accuracy, precision, recall, F1."""
    return {
        'accuracy' : accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='weighted', zero_division=0),
        'recall'   : recall_score(all_labels, all_preds,    average='weighted', zero_division=0),
        'f1'       : f1_score(all_labels, all_preds,        average='weighted', zero_division=0),
    }


def plot_training_curves(history, model_name):
    """Plot loss and accuracy curves side-by-side."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f'{model_name} — Training Curves', fontsize=14, fontweight='bold')

    axes[0].plot(history['train_loss'], label='Train', color='steelblue')
    axes[0].plot(history['val_loss'],   label='Val',   color='coral')
    axes[0].set_title('Loss');  axes[0].set_xlabel('Epoch');  axes[0].legend()

    axes[1].plot(history['train_acc'], label='Train', color='steelblue')
    axes[1].plot(history['val_acc'],   label='Val',   color='coral')
    axes[1].set_title('Accuracy');  axes[1].set_xlabel('Epoch');  axes[1].legend()

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(all_preds, all_labels, class_names, model_name):
    """Plot a labelled confusion matrix using seaborn."""
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} — Confusion Matrix', fontweight='bold')
    plt.ylabel('True Label');  plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()


CLASS_NAMES = full_frame_ds.classes   # ['fall', 'non-fall'] or similar
print('Utilities loaded ✓')

---
## 🔵 Model A — Simple CNN (Baseline)
A lightweight 3-block convolutional network built from scratch.  
Serves as the **baseline** to measure improvement from transfer learning.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  MODEL A — Simple CNN
# ═════════════════════════════════════════════════════════════════════════════

class SimpleCNN(nn.Module):
    """
    3-block custom CNN:
      Block 1: Conv(3→32) → BN → ReLU → MaxPool
      Block 2: Conv(32→64) → BN → ReLU → MaxPool
      Block 3: Conv(64→128) → BN → ReLU → MaxPool
      Head   : AdaptiveAvgPool → Flatten → FC(512) → Dropout → FC(2)
    """

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 224 → 112
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 112 → 56
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 56 → 28
        )
        self.pool = nn.AdaptiveAvgPool2d((4, 4))  # Fixed spatial output
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


# ── Instantiate & move to device ──────────────────────────────────────────────
cnn_model = SimpleCNN(num_classes=NUM_CLASSES).to(DEVICE)
print(cnn_model)
n_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

# ── Train ─────────────────────────────────────────────────────────────────────
cnn_model, cnn_history = full_training_loop(
    cnn_model, train_loader, val_loader,
    model_name='Simple CNN',
    save_path='best_simple_cnn.pth'
)

# ── Evaluate on Test Set ──────────────────────────────────────────────────────
_, test_acc_cnn, cnn_preds, cnn_labels = evaluate(cnn_model, test_loader)
cnn_metrics = compute_metrics(cnn_preds, cnn_labels)
RESULTS['Simple CNN'] = cnn_metrics

print('\n── Simple CNN — Test Results ──')
print(classification_report(cnn_labels, cnn_preds, target_names=CLASS_NAMES))
plot_training_curves(cnn_history, 'Simple CNN')
plot_confusion_matrix(cnn_preds, cnn_labels, CLASS_NAMES, 'Simple CNN')

---
## 🟢 Model B — ResNet18 (Transfer Learning)
Pre-trained on ImageNet-1K. Only the final fully-connected layer is replaced.  
The entire network is **fine-tuned** end-to-end at a low learning rate.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  MODEL B — ResNet18 (Transfer Learning)
# ═════════════════════════════════════════════════════════════════════════════

def build_resnet18(num_classes: int = NUM_CLASSES, freeze_backbone: bool = False):
    """
    Load ImageNet-pretrained ResNet18 and replace the FC head.

    Args:
        freeze_backbone: If True, freeze all layers except the final FC.
                         Set False here for full fine-tuning.
    """
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the classification head
    in_features = model.fc.in_features  # 512
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )
    return model


# ── Instantiate ───────────────────────────────────────────────────────────────
resnet_model = build_resnet18(freeze_backbone=False).to(DEVICE)
n_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f'ResNet18 — Trainable parameters: {n_params:,}')

# ── Train ─────────────────────────────────────────────────────────────────────
resnet_model, resnet_history = full_training_loop(
    resnet_model, train_loader, val_loader,
    model_name='ResNet18',
    save_path='best_resnet18.pth'
)

# ── Evaluate ──────────────────────────────────────────────────────────────────
_, _, resnet_preds, resnet_labels = evaluate(resnet_model, test_loader)
resnet_metrics = compute_metrics(resnet_preds, resnet_labels)
RESULTS['ResNet18'] = resnet_metrics

print('\n── ResNet18 — Test Results ──')
print(classification_report(resnet_labels, resnet_preds, target_names=CLASS_NAMES))
plot_training_curves(resnet_history, 'ResNet18')
plot_confusion_matrix(resnet_preds, resnet_labels, CLASS_NAMES, 'ResNet18')

---
## 🟡 Model C — MobileNetV2 (Efficient Transfer Learning)
Designed for edge deployment. Very fast inference thanks to depthwise-separable convolutions.  
Ideal if the Streamlit app needs to run on CPU.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  MODEL C — MobileNetV2 (Efficient Transfer Learning)
# ═════════════════════════════════════════════════════════════════════════════

def build_mobilenetv2(num_classes: int = NUM_CLASSES):
    """
    Load ImageNet-pretrained MobileNetV2 and attach a custom classifier.
    The inverted residual blocks are kept; only the final classifier is replaced.
    """
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

    # MobileNetV2 classifier: model.classifier is [Dropout, Linear(1280, 1000)]
    in_features = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes)
    )
    return model


# ── Instantiate ───────────────────────────────────────────────────────────────
mobilenet_model = build_mobilenetv2().to(DEVICE)
n_params = sum(p.numel() for p in mobilenet_model.parameters() if p.requires_grad)
print(f'MobileNetV2 — Trainable parameters: {n_params:,}')

# ── Train ─────────────────────────────────────────────────────────────────────
mobilenet_model, mobilenet_history = full_training_loop(
    mobilenet_model, train_loader, val_loader,
    model_name='MobileNetV2',
    save_path='best_mobilenetv2.pth'
)

# ── Evaluate ──────────────────────────────────────────────────────────────────
_, _, mobile_preds, mobile_labels = evaluate(mobilenet_model, test_loader)
mobile_metrics = compute_metrics(mobile_preds, mobile_labels)
RESULTS['MobileNetV2'] = mobile_metrics

print('\n── MobileNetV2 — Test Results ──')
print(classification_report(mobile_labels, mobile_preds, target_names=CLASS_NAMES))
plot_training_curves(mobilenet_history, 'MobileNetV2')
plot_confusion_matrix(mobile_preds, mobile_labels, CLASS_NAMES, 'MobileNetV2')

---
## 🔴 Model D — CNN + LSTM (Temporal Modelling)
A MobileNetV2 CNN **encoder** extracts per-frame features, which are fed as a sequence into a **bi-directional LSTM**.  
This lets the model reason about *motion dynamics* across time — crucial for distinguishing falls from non-falls.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  MODEL D — CNN + LSTM (Temporal Sequence Modelling)
# ═════════════════════════════════════════════════════════════════════════════

class CNNLSTMModel(nn.Module):
    """
    Architecture:
      1. CNN Encoder  : Pretrained MobileNetV2 feature extractor
                        Output per frame: 1280-dim vector
      2. LSTM         : Bidirectional 2-layer LSTM over T frames
      3. Classifier   : FC head on concatenated forward+backward hidden states

    Input  : (B, T, C, H, W)   — batch of video clips
    Output : (B, num_classes)  — class logits
    """

    def __init__(self, num_classes: int = NUM_CLASSES,
                 hidden_size: int = 256, num_layers: int = 2):
        super().__init__()

        # ── CNN Feature Extractor ─────────────────────────────────────────────
        backbone   = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        # Remove the classifier; keep only the feature layers
        self.cnn   = nn.Sequential(*list(backbone.children())[:-1],
                                    nn.AdaptiveAvgPool2d((1, 1)))
        self.cnn_out_dim = 1280  # MobileNetV2 last conv channels

        # Freeze CNN weights (optional — speeds training)
        for param in self.cnn.parameters():
            param.requires_grad = False

        # ── Temporal LSTM ────────────────────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size  = self.cnn_out_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            bidirectional = True,
            dropout     = 0.3 if num_layers > 1 else 0.0
        )

        # ── Classification Head ──────────────────────────────────────────────
        lstm_out_dim = hidden_size * 2  # bidirectional
        self.classifier = nn.Sequential(
            nn.LayerNorm(lstm_out_dim),
            nn.Dropout(0.4),
            nn.Linear(lstm_out_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape
        # Merge batch & time dims → (B*T, C, H, W) for CNN
        x = x.view(B * T, C, H, W)
        features = self.cnn(x)              # (B*T, 1280, 1, 1)
        features = features.view(B, T, -1)  # (B, T, 1280)
        # LSTM: output shape (B, T, hidden*2)
        lstm_out, _ = self.lstm(features)
        # Use the last time-step representation
        last_hidden = lstm_out[:, -1, :]    # (B, hidden*2)
        logits      = self.classifier(last_hidden)
        return logits


# ── Instantiate ───────────────────────────────────────────────────────────────
cnn_lstm_model = CNNLSTMModel(num_classes=NUM_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in cnn_lstm_model.parameters() if p.requires_grad)
print(f'CNN+LSTM — Trainable parameters: {n_params:,}')

# ── Custom train/eval wrappers (sequences have shape B,T,C,H,W) ───────────────
# The shared helpers already handle arbitrary input shapes,
# so we simply pass seq_train_loader / seq_val_loader.

cnn_lstm_model, lstm_history = full_training_loop(
    cnn_lstm_model, seq_train_loader, seq_val_loader,
    model_name='CNN + LSTM',
    save_path='best_cnn_lstm.pth'
)

# ── Evaluate ──────────────────────────────────────────────────────────────────
_, _, lstm_preds, lstm_labels = evaluate(cnn_lstm_model, seq_test_loader)
lstm_metrics = compute_metrics(lstm_preds, lstm_labels)
RESULTS['CNN + LSTM'] = lstm_metrics

print('\n── CNN + LSTM — Test Results ──')
print(classification_report(lstm_labels, lstm_preds, target_names=CLASS_NAMES))
plot_training_curves(lstm_history, 'CNN + LSTM')
plot_confusion_matrix(lstm_preds, lstm_labels, CLASS_NAMES, 'CNN + LSTM')

---
## 📊 4. Comparative Results Table & Model Selection

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Build & display comparison table
# ─────────────────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(RESULTS).T.reset_index()
results_df.columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
results_df = results_df.sort_values('F1-Score', ascending=False).reset_index(drop=True)

# Format as percentages for readability
fmt_df = results_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    fmt_df[col] = fmt_df[col].map(lambda v: f'{v*100:.2f}%')

print('\n' + '═'*65)
print('  COMPARATIVE RESULTS (Test Set — sorted by F1-Score)'.center(65))
print('═'*65)
print(fmt_df.to_string(index=False))
print('═'*65)

# ── Bar-chart comparison ──────────────────────────────────────────────────────
metrics    = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_names = results_df['Model'].tolist()

x = np.arange(len(metrics))
width = 0.18
colours = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (model, colour) in enumerate(zip(model_names, colours)):
    vals = [results_df.loc[results_df['Model'] == model, m].values[0] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=model, color=colour, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Fall Detection — Model Comparison (Test Set)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# ── Select & save the best model ──────────────────────────────────────────────
best_model_name = results_df.iloc[0]['Model']
MODEL_MAP = {
    'Simple CNN' : (cnn_model,       'best_simple_cnn.pth'),
    'ResNet18'   : (resnet_model,    'best_resnet18.pth'),
    'MobileNetV2': (mobilenet_model, 'best_mobilenetv2.pth'),
    'CNN + LSTM' : (cnn_lstm_model,  'best_cnn_lstm.pth'),
}
best_model_obj, best_model_path = MODEL_MAP[best_model_name]

# Save as the canonical 'best model' used by the Streamlit app
import json
meta = {
    'model_name'  : best_model_name,
    'model_path'  : 'best_model.pth',
    'class_names' : CLASS_NAMES,
    'img_size'    : IMG_SIZE,
    'num_frames'  : NUM_FRAMES,
    'is_sequence' : best_model_name == 'CNN + LSTM',
}
torch.save(best_model_obj.state_dict(), 'best_model.pth')
with open('model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'\n🏆 Best Model: {best_model_name}')
print(f'   F1-Score : {results_df.iloc[0]["F1-Score"]*100:.2f}%')
print('   Saved → best_model.pth + model_meta.json ✓')